In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#444444'
plt.rcParams['figure.dpi'] = 150

COLOR_PRIMARY = '#1f4e79'
COLOR_ACCENT = '#d95f02'
COLOR_ACCENT2 = '#1b9e77'

### LOAD & CLEAN

In [4]:
df = pd.read_csv('Unemployment in India.csv')
df.columns = [c.strip() for c in df.columns]

df = df.dropna(how='all')

for col in ['Region', 'Frequency', 'Area']:
    df[col] = df[col].astype(str).str.strip()

df = df.dropna(subset=['Date', 'Estimated Unemployment Rate (%)'])

df['Date'] = pd.to_datetime(df['Date'].astype(str).str.strip(), format='%d-%m-%Y', errors='coerce')
df = df.dropna(subset=['Date'])

df = df.rename(columns={
    'Estimated Unemployment Rate (%)': 'UnemploymentRate',
    'Estimated Employed': 'Employed',
    'Estimated Labour Participation Rate (%)': 'LabourParticipationRate'
})

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['MonthName'] = df['Date'].dt.strftime('%b')
df['YearMonth'] = df['Date'].dt.to_period('M')

print("Cleaned shape:", df.shape)
print("Date range:", df['Date'].min(), "to", df['Date'].max())
print("Regions:", df['Region'].nunique())
print(df.head())
df.to_csv('cleaned_unemployment.csv', index=False)

Cleaned shape: (740, 11)
Date range: 2019-05-31 00:00:00 to 2020-06-30 00:00:00
Regions: 28
           Region       Date Frequency  UnemploymentRate    Employed  \
0  Andhra Pradesh 2019-05-31   Monthly              3.65  11999139.0   
1  Andhra Pradesh 2019-06-30   Monthly              3.05  11755881.0   
2  Andhra Pradesh 2019-07-31   Monthly              3.75  12086707.0   
3  Andhra Pradesh 2019-08-31   Monthly              3.32  12285693.0   
4  Andhra Pradesh 2019-09-30   Monthly              5.17  12256762.0   

   LabourParticipationRate   Area  Year  Month MonthName YearMonth  
0                    43.24  Rural  2019      5       May   2019-05  
1                    42.05  Rural  2019      6       Jun   2019-06  
2                    43.50  Rural  2019      7       Jul   2019-07  
3                    43.97  Rural  2019      8       Aug   2019-08  
4                    44.68  Rural  2019      9       Sep   2019-09  


### KEY SUMMARY 

In [5]:
overall_mean = df['UnemploymentRate'].mean()
overall_median = df['UnemploymentRate'].median()
overall_max = df['UnemploymentRate'].max()
overall_max_row = df.loc[df['UnemploymentRate'].idxmax()]

pre_covid = df[df['Date'] < '2020-03-01']['UnemploymentRate'].mean()
covid_peak_period = df[(df['Date'] >= '2020-03-01') & (df['Date'] <= '2020-06-30')]['UnemploymentRate'].mean()
lockdown_only = df[(df['Date'] >= '2020-04-01') & (df['Date'] <= '2020-05-31')]['UnemploymentRate'].mean()
early_reopening = df[df['Date'] == '2020-06-30']['UnemploymentRate'].mean()

print("\nSUMMARY ")
print(f"Overall mean unemployment: {overall_mean:.2f}%")
print(f"Overall median unemployment: {overall_median:.2f}%")
print(f"Max unemployment: {overall_max:.2f}% in {overall_max_row['Region']} ({overall_max_row['Date'].strftime('%b %Y')})")
print(f"Pre-COVID (before Mar 2020) avg: {pre_covid:.2f}%")
print(f"COVID peak (Mar-Jun 2020) avg: {covid_peak_period:.2f}%")
print(f"Strict lockdown (Apr-May 2020) avg: {lockdown_only:.2f}%")
print(f"Early reopening (Jun 2020) avg: {early_reopening:.2f}%")


SUMMARY 
Overall mean unemployment: 11.79%
Overall median unemployment: 8.35%
Max unemployment: 76.74% in Puducherry (Apr 2020)
Pre-COVID (before Mar 2020) avg: 9.51%
COVID peak (Mar-Jun 2020) avg: 17.77%
Strict lockdown (Apr-May 2020) avg: 24.26%
Early reopening (Jun 2020) avg: 11.90%


CHART 1: National monthly trend with COVID annotation

In [6]:
monthly_national = df.groupby('YearMonth')['UnemploymentRate'].mean().reset_index()
monthly_national['YearMonth_ts'] = monthly_national['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(monthly_national['YearMonth_ts'], monthly_national['UnemploymentRate'],
        marker='o', color=COLOR_PRIMARY, linewidth=2.5, markersize=5, zorder=3)
ax.fill_between(monthly_national['YearMonth_ts'], monthly_national['UnemploymentRate'],
                 color=COLOR_PRIMARY, alpha=0.08)

ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-30'), color=COLOR_ACCENT, alpha=0.15,
           label='National Lockdown (Mar-Jun 2020)')

peak_row = monthly_national.loc[monthly_national['UnemploymentRate'].idxmax()]
ax.annotate(f"Peak: {peak_row['UnemploymentRate']:.1f}%\n{peak_row['YearMonth_ts'].strftime('%b %Y')}",
            xy=(peak_row['YearMonth_ts'], peak_row['UnemploymentRate']),
            xytext=(20, 25), textcoords='offset points',
            fontsize=10, fontweight='bold', color=COLOR_ACCENT,
            arrowprops=dict(arrowstyle='->', color=COLOR_ACCENT, lw=1.5))

ax.set_title('India National Average Unemployment Rate Over Time', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Unemployment Rate (%)', fontsize=11)
ax.legend(loc='upper right', frameon=True)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('1_national_trend.png', bbox_inches='tight')
plt.close()

CHART 2: Rural vs Urban comparison over time

In [7]:
area_monthly = df.groupby(['YearMonth', 'Area'])['UnemploymentRate'].mean().reset_index()
area_monthly['YearMonth_ts'] = area_monthly['YearMonth'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(11, 5.5))
for area, color in [('Rural', COLOR_ACCENT2), ('Urban', COLOR_ACCENT)]:
    sub = area_monthly[area_monthly['Area'] == area]
    ax.plot(sub['YearMonth_ts'], sub['UnemploymentRate'], marker='o', label=area,
            linewidth=2.2, markersize=4, color=color)

ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-30'), color='grey', alpha=0.12)
ax.set_title('Rural vs Urban Unemployment Rate Trend', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Unemployment Rate (%)', fontsize=11)
ax.legend(title='Area', loc='upper right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig('2_rural_urban_trend.png', bbox_inches='tight')
plt.close()

CHART 3: Top / Bottom 10 states by average unemployment

In [8]:
state_avg = df.groupby('Region')['UnemploymentRate'].mean().sort_values(ascending=False)
top10 = state_avg.head(10)
bottom10 = state_avg.tail(10).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].barh(top10.index[::-1], top10.values[::-1], color=COLOR_ACCENT)
axes[0].set_title('Top 10 States - Highest Avg. Unemployment', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Avg. Unemployment Rate (%)')
for i, v in enumerate(top10.values[::-1]):
    axes[0].text(v + 0.2, i, f"{v:.1f}%", va='center', fontsize=9)

axes[1].barh(bottom10.index[::-1], bottom10.values[::-1], color=COLOR_ACCENT2)
axes[1].set_title('Top 10 States - Lowest Avg. Unemployment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Avg. Unemployment Rate (%)')
for i, v in enumerate(bottom10.values[::-1]):
    axes[1].text(v + 0.1, i, f"{v:.1f}%", va='center', fontsize=9)

plt.suptitle('State-wise Unemployment Rate Comparison (Full Period Average)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('3_state_comparison.png', bbox_inches='tight')
plt.close()

CHART 4: Pre vs COVID-peak vs Post bar chart

In [9]:
phases = ['Pre-COVID\n(May 2019-Feb 2020)', 'Lockdown\n(Apr-May 2020)', 'Early Reopening\n(Jun 2020)']
values = [pre_covid, lockdown_only, early_reopening]
colors_phase = [COLOR_PRIMARY, COLOR_ACCENT, COLOR_ACCENT2]

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.bar(phases, values, color=colors_phase, width=0.55)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.4, f"{v:.1f}%", ha='center', fontsize=12, fontweight='bold')
ax.set_title('Impact of COVID-19 Lockdown on Average Unemployment Rate', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Unemployment Rate (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
plt.tight_layout()
plt.savefig('4_covid_impact.png', bbox_inches='tight')
plt.close()

CHART 5: Seasonal pattern - average by calendar month

In [11]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df_precovid = df[df['Date'] < '2020-03-01']
monthly_seasonal_excl = df_precovid.groupby('MonthName')['UnemploymentRate'].mean().reindex(month_order)
monthly_seasonal_excl = monthly_seasonal_excl.dropna()

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(monthly_seasonal_excl.index, monthly_seasonal_excl.values, color=COLOR_PRIMARY)
for bar, v in zip(bars, monthly_seasonal_excl.values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.1, f"{v:.1f}%", ha='center', fontsize=9)
ax.set_title('Seasonal Pattern: Avg. Unemployment by Calendar Month (Pre-COVID, May 2019-Feb 2020)',
             fontsize=12.5, fontweight='bold', pad=15)
ax.set_ylabel('Unemployment Rate (%)', fontsize=11)
ax.set_xlabel('Month', fontsize=11)
plt.tight_layout()
plt.savefig('5_seasonal_pattern.png', bbox_inches='tight')
plt.close()

CHART 6: Heatmap - state x month unemployment

In [12]:
pivot = df.pivot_table(index='Region', columns='YearMonth', values='UnemploymentRate', aggfunc='mean')
pivot.columns = [str(c) for c in pivot.columns]

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(pivot, cmap='RdYlGn_r', ax=ax, cbar_kws={'label': 'Unemployment Rate (%)'}, linewidths=0.3, linecolor='white')
ax.set_title('Unemployment Rate Heatmap: State vs Month', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('State/UT')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('6_heatmap.png', bbox_inches='tight')
plt.close()

CHART 7: Labour Participation Rate vs Unemployment scatter

In [13]:
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(df['LabourParticipationRate'], df['UnemploymentRate'],
                 c=df['Year'], cmap='viridis', alpha=0.6, s=35)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Year')
ax.set_title('Labour Participation Rate vs Unemployment Rate', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Estimated Labour Participation Rate (%)')
ax.set_ylabel('Estimated Unemployment Rate (%)')
plt.tight_layout()
plt.savefig('7_participation_scatter.png', bbox_inches='tight')
plt.close()

Additional Stats for Report

In [14]:
area_avg = df.groupby('Area')['UnemploymentRate'].mean()
print("\nArea averages:\n", area_avg)

covid_state_impact = df[(df['Date']>='2020-03-01')&(df['Date']<='2020-06-30')].groupby('Region')['UnemploymentRate'].mean().sort_values(ascending=False)
print("\nTop 5 states hit hardest during COVID peak:\n", covid_state_impact.head(5))

max_month_seasonal = monthly_seasonal_excl.idxmax()
min_month_seasonal = monthly_seasonal_excl.idxmin()
print(f"\nSeasonal peak month (excl 2020): {max_month_seasonal} ({monthly_seasonal_excl.max():.2f}%)")
print(f"Seasonal low month (excl 2020): {min_month_seasonal} ({monthly_seasonal_excl.min():.2f}%)")

print("\nCharts generated successfully.")


Area averages:
 Area
Rural    10.324791
Urban    13.166614
Name: UnemploymentRate, dtype: float64

Top 5 states hit hardest during COVID peak:
 Region
Puducherry    38.95500
Jharkhand     36.34875
Haryana       34.65250
Bihar         31.63125
Tripura       26.70250
Name: UnemploymentRate, dtype: float64

Seasonal peak month (excl 2020): Feb (9.96%)
Seasonal low month (excl 2020): May (8.87%)

Charts generated successfully.
